# EDA y Limpieza de Datos

**TFM:** Sistema de Apoyo a la Decisión Clínica en Oncología Pediátrica  
**Autor:** Alonso Castañón González  
**Dataset:** SEER Research Data — Osteosarcoma y Sarcoma de Ewing pediátrico (2000–2023)

**Objetivo de este notebook:** Realizar la carga, inspección inicial, limpieza y preparación del dataset de SEER
para su uso en el modelo predictivo de supervivencia.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

SEED = 2026
np.random.seed(SEED)

print("Entorno configurado correctamente")

Entorno configurado correctamente


## 1) Carga de datos

In [2]:
# Ruta del dataset del SEER
DATA_PATH = '../data/seer_osteosarcoma_ewing.csv'

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head(5)

Shape: (4436, 14)
Filas: 4,436
Columnas: 14


,Age recode with <1 year olds and 90+,Sex,Year of diagnosis,Histologic Type ICD-O-3,Primary Site,Combined Summary Stage with Expanded Regional Codes (2004+),CS tumor size (2004-2015),RX Summ--Surg Prim Site (1998-2022),Radiation recode,"Chemotherapy recode (yes, no/unk)",Survival months,Survival months flag,SEER cause-specific death classification,Vital status recode (study cutoff used)
0,05-09 years,Female,2004,9260,400,Regional by direct extension only,064,25,Beam radiation,Yes,239,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
1,15-19 years,Male,2001,9260,413,Blank(s),Blank(s),30,None/Unknown,Yes,275,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
2,10-14 years,Female,2001,9260,414,Blank(s),Blank(s),00,Beam radiation,Yes,119,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead
3,10-14 years,Female,2001,9180,402,Blank(s),Blank(s),30,None/Unknown,Yes,274,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
4,15-19 years,Female,2000,9181,414,Blank(s),Blank(s),00,Beam radiation,Yes,23,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead


## 2) Inspección inicial del dataset

In [3]:
# Tipos de datos y valores no nulos por columna
print("---- TIPOS DE DATOS ----")
print(df.dtypes)
print(f"\n---- VALORES NO NULOS ----")
print(df.notnull().sum())

---- TIPOS DE DATOS ----
Age recode with <1 year olds and 90+                           object
Sex                                                            object
Year of diagnosis                                               int64
Histologic Type ICD-O-3                                         int64
Primary Site                                                    int64
Combined Summary Stage with Expanded Regional Codes (2004+)    object
CS tumor size (2004-2015)                                      object
RX Summ--Surg Prim Site (1998-2022)                            object
Radiation recode                                               object
Chemotherapy recode (yes, no/unk)                              object
Survival months                                                 int64
Survival months flag                                           object
SEER cause-specific death classification                       object
Vital status recode (study cutoff used)                        ob

## 3) Renombrado de columnas

In [4]:
# Renombrado de columnas para facilitar el trabajo y la identificación
column_mapping = {
    'Age recode with <1 year olds and 90+': 'age_group',
    'Sex': 'sex',
    'Year of diagnosis': 'year_diagnosis',
    'Histologic Type ICD-O-3': 'histology_code',
    'Primary Site': 'primary_site',
    'Combined Summary Stage with Expanded Regional Codes (2004+)': 'stage',
    'CS tumor size (2004-2015)': 'tumor_size_cs',
    'RX Summ--Surg Prim Site (1998-2022)': 'surgery_code',
    'Radiation recode': 'radiation',
    'Chemotherapy recode (yes, no/unk)': 'chemotherapy',
    'Survival months': 'survival_months',
    'Survival months flag': 'survival_months_flag',
    'SEER cause-specific death classification': 'cause_specific_death',
    'Vital status recode (study cutoff used)': 'vital_status'
}

df = df.rename(columns=column_mapping)

print("Columnas renombradas:")
print(df.columns.tolist())

df.head(5)

Columnas renombradas:
['age_group', 'sex', 'year_diagnosis', 'histology_code', 'primary_site', 'stage', 'tumor_size_cs', 'surgery_code', 'radiation', 'chemotherapy', 'survival_months', 'survival_months_flag', 'cause_specific_death', 'vital_status']


,age_group,sex,year_diagnosis,histology_code,primary_site,stage,tumor_size_cs,surgery_code,radiation,chemotherapy,survival_months,survival_months_flag,cause_specific_death,vital_status
0,05-09 years,Female,2004,9260,400,Regional by direct extension only,064,25,Beam radiation,Yes,239,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
1,15-19 years,Male,2001,9260,413,Blank(s),Blank(s),30,None/Unknown,Yes,275,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
2,10-14 years,Female,2001,9260,414,Blank(s),Blank(s),00,Beam radiation,Yes,119,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead
3,10-14 years,Female,2001,9180,402,Blank(s),Blank(s),30,None/Unknown,Yes,274,Complete dates are available and there are mor...,Alive or dead of other cause,Alive
4,15-19 years,Female,2000,9181,414,Blank(s),Blank(s),00,Beam radiation,Yes,23,Complete dates are available and there are mor...,Dead (attributable to this cancer dx),Dead


## 4) Detección real de valores missing (Blank(s)) y reemplazo por NaN

In [5]:
# Se comprueba la posible existencia de valores perdidos.

missing = df.isnull().sum()
print("---- VALORES MISSING POR COLUMNA INICIALES ----")
print(missing)

---- VALORES MISSING POR COLUMNA INICIALES ----
age_group               0
sex                     0
year_diagnosis          0
histology_code          0
primary_site            0
stage                   0
tumor_size_cs           0
surgery_code            0
radiation               0
chemotherapy            0
survival_months         0
survival_months_flag    0
cause_specific_death    0
vital_status            0
dtype: int64


SEER codifica los missing como el string `Blank(s)`, no como NaN. Necesitamos detectarlos antes de cualquier transformación.

Los valores `None/Unknown` se mantienen como categoría explícita ya que representan casos donde el tratamiento existía pero no fue registrado, lo cual puede ser información clínicamente relevante y distinta de un dato ausente.

In [6]:
# Se identifican los valores desconocidos codificados como Blank(s)

BLANK = 'Blank(s)'

print("---- VALORES BLANK(S) POR COLUMNA ----")
for col in df.columns:
    n_blanks = (df[col].astype(str).str.strip() == BLANK).sum()
    pct = n_blanks / len(df) * 100
    print(f"{col}: {n_blanks} ({pct:.1f}%)")

---- VALORES BLANK(S) POR COLUMNA ----
age_group: 0 (0.0%)
sex: 0 (0.0%)
year_diagnosis: 0 (0.0%)
histology_code: 0 (0.0%)
primary_site: 0 (0.0%)
stage: 727 (16.4%)
tumor_size_cs: 2115 (47.7%)
surgery_code: 122 (2.8%)
radiation: 0 (0.0%)
chemotherapy: 0 (0.0%)
survival_months: 0 (0.0%)
survival_months_flag: 0 (0.0%)
cause_specific_death: 0 (0.0%)
vital_status: 0 (0.0%)


### Conclusiones sobre valores missing

- **CS tumor size (2004-2015)**: 47.7% de missing. Es la variable más problemática.
  Solo tiene datos entre 2004-2015 por diseño del registro SEER. Se conserva en el
  dataset pero **no se usará como predictor en el modelo ML** dado su alto porcentaje
  de missing y su cobertura temporal parcial. Se pueden hacer pruebas con solo la parte
  del dataset que tiene estos valores para ver como afecta su inclusión en las predicciones.

- **Combined Summary Stage (2004+)**: 16.4% de missing. El missing se concentra
  en los años 2000-2003, anteriores al sistema de estadificación combinada.
  Se usará en el modelo imputando o excluyendo estos casos según el análisis posterior.

- **RX Summ--Surg Prim Site (1998-2022)**: 2.8% de missing. Porcentaje bajo y
  manejable. Se imputará o se excluirán estas filas en el pipeline de modelado.

- El resto de columnas están completas al 100%.

In [7]:
# Se reemplaza el string "Blank(s)" por NaN real de pandas.

df = df.replace('Blank(s)', np.nan)

# Se verifica si ahora detecta bien los valores perdidos.
print("---- MISSING VALUES REALES (NaN) POR COLUMNA ----")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'missing': missing,
    'porcentaje': missing_pct
}).query('missing > 0')

print(missing_df)
print(f"\nTotal filas del dataset: {len(df)}")
print(f"Filas completas sin ningún NaN: {df.dropna().shape[0]}")

---- MISSING VALUES REALES (NaN) POR COLUMNA ----
               missing  porcentaje
stage              727        16.4
tumor_size_cs     2115        47.7
surgery_code       122         2.8

Total filas del dataset: 4436
Filas completas sin ningún NaN: 2321


Es importante señalar que los valores perdidos en *stage* y *tumor_size_cs* no siguen un patrón aleatorio (MCAR, Missing Completely At Random), sino que están sistemáticamente asociados al año de diagnóstico del paciente, lo que se corresponde con un mecanismo MAR (Missing At Random) condicionado a esta variable temporal. En el caso de *tumor_size_cs*, la ausencia de valores se concentra en el periodo 2004-2015 debido a cambios en los protocolos de codificación del registro SEER; de forma similar, los nulos en *stage* se concentran en 2000-2003, años previos a la adopción del sistema de estadificación combinada (Combined Summary Stage). Este origen estructural del missingness, ligado a la evolución metodológica del propio registro y no a características clínicas de los pacientes, justifica el tratamiento diferenciado por variable adoptado en este trabajo (exclusión como predictor primario, imputación condicionada o análisis por subconjuntos temporales) frente a una eliminación indiscriminada de filas (dropna global), que supondría descartar cerca de la mitad de la muestra disponible.

## 5) Clasificación de columnas, conversión de tipos y mapeo de códigos

### 5.1) Inspección de valores reales por columna

In [8]:
print("---- VALORES ÚNICOS POR COLUMNA ----")
for col in df.columns:
    n_unique = df[col].nunique()
    sample = df[col].dropna().unique()
    print(f"\n{col} (dtype: {df[col].dtype}) — {n_unique} valores únicos")
    print(f"  Valores:\n {sample}")

---- VALORES ÚNICOS POR COLUMNA ----

age_group (dtype: object) — 5 valores únicos
  Valores:
 ['05-09 years' '15-19 years' '10-14 years' '00 years' '01-04 years']

sex (dtype: object) — 2 valores únicos
  Valores:
 ['Female' 'Male']

year_diagnosis (dtype: int64) — 24 valores únicos
  Valores:
 [2004 2001 2000 2002 2003 2005 2006 2007 2012 2008 2009 2010 2011 2022
 2013 2014 2015 2020 2016 2017 2018 2019 2021 2023]

histology_code (dtype: int64) — 9 valores únicos
  Valores:
 [9260 9180 9181 9186 9182 9183 9187 9185 9184]

primary_site (dtype: int64) — 60 valores únicos
  Valores:
 [400 413 414 402 412 493 403 410 480 495 491 492 409 714 419 490 496  19
 720 649 749 761 401 494 519 418 715 548 411 809 311 382 719 499 712 445
 529 696 408  79 721 220 384 501 343 711 109 310 475 380 341 569 471 621
 701 729 381 713 509 619]

stage (dtype: object) — 6 valores únicos
  Valores:
 ['Regional by direct extension only' 'Distant site(s)/node(s) involved'
 'Localized only'
 'Regional by both di

### 5.2) Conclusiones de la inspección y plan de acción

Tras observar los valores iniciales, se toman las siguientes decisiones:

| Columna | Tipo actual | Tipo real | Acción |
|---|---|---|---|
| `age_group` | object | Categórica ordinal | Convertir a Categorical con orden (puede ser útil para visualizaciones y algunos modelos)|
| `sex` | object | Categórica nominal | Convertir a Categorical |
| `year_diagnosis` | int64 | Numérica discreta | Mantener como int64 |
| `histology_code` | int64 | Categórica nominal (código) | Mapear a nombre de tumor (publicados en el diccionario ICD-O-3 por la OMS y SEER) |
| `primary_site` | int64 | Categórica nominal (código) | Mapear a localización anatómica simplificada (códigos C40-C41 del ICD-O-3 o agrupación por categorías clínicas como huesos largos, huesos planos, pelvis, etc. ???) |
| `stage` | object | Categórica ordinal | Limpiar categorías y convertir a Categorical con orden |
| `tumor_size_cs` | object | Numérica continua (mm) | Convertir a float, tratar códigos especiales (999, 997, 000...) |
| `surgery_code` | object | Categórica nominal (código) | Mapear a descripción de cirugía (publicadas en seer.cancer.gov/tools/surgery) |
| `radiation` | object | Categórica nominal | Simplificar categorías |
| `chemotherapy` | object | Categórica nominal | Convertir a Categorical |
| `survival_months` | int64 | Numérica continua | Se mantiene |
| `survival_months_flag` | object | Categórica nominal | Simplificar a código corto |
| `cause_specific_death` | object | Categórica nominal | Simplificar categorías |
| `vital_status` | object | Categórica nominal | Convertir a Categorical |

### 5.3) Conversión de tipos y transformaciones por columna

#### age_group

In [9]:
# Categórica ordinal con orden natural de edad
age_order = ['00 years', '01-04 years', '05-09 years', '10-14 years', '15-19 years']

df['age_group'] = pd.Categorical(df['age_group'], categories=age_order, ordered=True)

print(df['age_group'].dtype)
print(df['age_group'].cat.categories)
print(df['age_group'].value_counts().sort_index())

category
Index(['00 years', '01-04 years', '05-09 years', '10-14 years', '15-19 years'], dtype='object')
age_group
00 years         26
01-04 years     176
05-09 years     757
10-14 years    1746
15-19 years    1731
Name: count, dtype: int64


#### sex

In [10]:
# Categórica nominal con dos valores
df['sex'] = pd.Categorical(df['sex'])

print(df['sex'].dtype)
print(df['sex'].value_counts())

category
sex
Male      2558
Female    1878
Name: count, dtype: int64


#### year_diagnosis

In [11]:
# Numérica discreta, se verifica el rango

print(f"Rango: {df['year_diagnosis'].min()} — {df['year_diagnosis'].max()}")
print(f"Dtype: {df['year_diagnosis'].dtype}")

Rango: 2000 — 2023
Dtype: int64


#### histology_code

Fuente: National Cancer Institute, Surveillance, Epidemiology, and End Results Program. ICD-O-3 SEER Site/Histology Validation List. April 29, 2022.
Disponible en: seer.cancer.gov/icd-o-3/

In [ ]:
# Código ICD-O-3 numérico que se mapea a nombre de tumor mediante el diccionario ICD-O-3 OMS/SEER

histology_map = {
    9180: 'Osteosarcoma NOS',
    9181: 'Chondroblastic osteosarcoma',
    9182: 'Fibroblastic osteosarcoma',
    9183: 'Telangiectatic osteosarcoma',
    9184: 'Osteosarcoma in Paget disease',
    9185: 'Small cell osteosarcoma',
    9186: 'Central osteosarcoma',
    9187: 'Intraosseous well differentiated osteosarcoma',
    9260: 'Ewing sarcoma'
}

df['tumor_type'] = df['histology_code'].map(histology_map)
df['tumor_type'] = pd.Categorical(df['tumor_type'])

print(df['tumor_type'].value_counts())
print(f"\nCódigos sin mapear: {df['tumor_type'].isnull().sum()}")

tumor_type
Osteosarcoma NOS                                 2058
Ewing sarcoma                                    1649
Chondroblastic osteosarcoma                       410
Telangiectatic osteosarcoma                       108
Central osteosarcoma                              103
Fibroblastic osteosarcoma                          76
Small cell osteosarcoma                            25
Intraosseous well differentiated osteosarcoma       6
Osteosarcoma in Paget disease                       1
Name: count, dtype: int64

Códigos sin mapear: 0


Los subtipos minoritarios de osteosarcoma (telangiectásico, condroblástico, fibroblástico, etc.) presentan un número de casos muy reducido en comparación con "Osteosarcoma NOS". Se mantienen como categorías independientes en esta fase. En el notebook de modelado se evaluará el impacto de agruparlos bajo una única categoría "Osteosarcoma" comparando el rendimiento del modelo en ambos escenarios.

#### primary_site

Los 60 códigos de localización anatómica se agrupan en 8 categorías clínicas siguiendo el estándar usado en estudios poblacionales con datos SEER para tumores óseos pediátricos (Chen et al., 2024; Vasudeva et al., 2019). En lugar de mapear cada código con su hueso exacto, lo que haría tener muchas categorías muy poco representadas, se decide realizar esta agrupación ya que garantiza suficiente representación estadística en cada categoría para el modelo.

La correspondencia entre códigos numéricos y localizaciones anatómicas se obtiene del SEER Summary Stage 2018 Coding Manual (capítulo Bone), donde los códigos C40.0-C41.9 del ICD-O-3 se almacenan en SEER sin la "C" y sin el punto (C40.2 → 402, C41.4 → 414, etc.).

Fuente: National Cancer Institute, SEER Program. *Summary Stage 2018 Coding
Manual — Bone chapter*. November 2024.
Available at: seer.cancer.gov/tools/ssm/SSM2018-BONE.pdf

In [ ]:
# Agrupación clínica de primary_site en 8 categorías

site_map_grouped = {
    # Extremidades superiores
    400: 'Upper limb bones',
    401: 'Upper limb bones',
    408: 'Upper limb bones',
    471: 'Upper limb bones',
    491: 'Upper limb bones',
    # Extremidades inferiores
    402: 'Lower limb bones',
    403: 'Lower limb bones',
    475: 'Lower limb bones',
    492: 'Lower limb bones',
    # Pelvis / cadera
    414: 'Pelvis / hip',
    495: 'Pelvis / hip',
    380: 'Pelvis / hip',
    381: 'Pelvis / hip',
    382: 'Pelvis / hip',
    384: 'Pelvis / hip',
    548: 'Pelvis / hip',
    # Tronco / costillas / esternón
    413: 'Trunk / ribs / sternum',
    493: 'Trunk / ribs / sternum',
    496: 'Trunk / ribs / sternum',
    # Columna vertebral
    412: 'Vertebral column',
    749: 'Vertebral column',
    # Cráneo / cabeza / mandíbula
    410: 'Skull / head / jaw',
    411: 'Skull / head / jaw',
    310: 'Skull / head / jaw',
    311: 'Skull / head / jaw',
    341: 'Skull / head / jaw',
    343: 'Skull / head / jaw',
    490: 'Skull / head / jaw',
    445: 'Skull / head / jaw',
    # Sistema nervioso central / meninges
    701: 'CNS / meninges',
    711: 'CNS / meninges',
    712: 'CNS / meninges',
    713: 'CNS / meninges',
    714: 'CNS / meninges',
    715: 'CNS / meninges',
    719: 'CNS / meninges',
    720: 'CNS / meninges',
    721: 'CNS / meninges',
    729: 'CNS / meninges',
    761: 'CNS / meninges',
    # Otros / no especificados
    409: 'Other / NOS',
    418: 'Other / NOS',
    419: 'Other / NOS',
    480: 'Other / NOS',
    494: 'Other / NOS',
    499: 'Other / NOS',
    519: 'Other / NOS',
    529: 'Other / NOS',
    696: 'Other / NOS',
    809: 'Other / NOS',
    19:  'Other / NOS',
    79:  'Other / NOS',
    109: 'Other / NOS',
    220: 'Other / NOS',
    501: 'Other / NOS',
    509: 'Other / NOS',
    569: 'Other / NOS',
    619: 'Other / NOS',
    621: 'Other / NOS',
    649: 'Other / NOS',
}

df['primary_site'] = df['primary_site'].map(site_map_grouped)
df['primary_site'] = pd.Categorical(df['primary_site'])

print(df['primary_site'].value_counts())
print(f"\nCategorías: {df['primary_site'].nunique()}")

primary_site
Lower limb bones          2595
Upper limb bones           579
Pelvis / hip               469
Trunk / ribs / sternum     317
Skull / head / jaw         200
Vertebral column           151
Other / NOS                101
CNS / meninges              24
Name: count, dtype: int64

Categorías: 8


#### stage

La variable `stage` presenta 6 valores únicos, de los cuales uno es `Unknown/unstaged/unspecified/DCO`. A diferencia de los valores `None/Unknown` en variables como `radiation` o `surgery_code`, donde el desconocimiento del tratamiento es información clínicamente relevante en sí misma, en este caso el valor desconocido se debe a un cambio de protocolo en el registro SEER: el sistema de estadificación combinada (Combined Summary Stage) no estaba implementado antes de 2004, por lo que los casos de 2000-2003 no pudieron ser estadificados con este sistema. Por tanto, este valor no aporta información clínica diferencial y se convierte a NaN.

El orden clínico de las categorías restantes refleja la progresión de la enfermedad de menor a mayor severidad:

1. `Localized only`: tumor confinado al hueso de origen
2. `Regional by direct extension only`: extensión directa a tejidos adyacentes
3. `Regional lymph nodes involved only`: afectación de ganglios linfáticos regionales
4. `Regional by both direct extension and lymph node involvement`: extensión directa y ganglionar simultánea
5. `Distant site(s)/node(s) involved`: metástasis a distancia

In [14]:
# Categórica ordinal

# Se convierte Unknown/unstaged/unspecified/DCO a NaN
df['stage'] = df['stage'].replace('Unknown/unstaged/unspecified/DCO', np.nan)

# Se define orden clínico de menor a mayor severidad
stage_order = [
    'Localized only',
    'Regional by direct extension only',
    'Regional lymph nodes involved only',
    'Regional by both direct extension and lymph node involvement',
    'Distant site(s)/node(s) involved'
]

df['stage'] = pd.Categorical(df['stage'], categories=stage_order, ordered=True)

print(df['stage'].value_counts(dropna=False).sort_index())
print(f"\nNaN totales en stage: {df['stage'].isnull().sum()}")

stage
Localized only                                                  1258
Regional by direct extension only                               1251
Regional lymph nodes involved only                                20
Regional by both direct extension and lymph node involvement      40
Distant site(s)/node(s) involved                                 989
NaN                                                              878
Name: count, dtype: int64

NaN totales en stage: 878


#### tumor_size_cs

El campo CS Tumor Size usa un sistema de 3 dígitos donde los valores 001-988 representan el tamaño real del tumor en **milímetros**. Los códigos especiales no representan tamaños reales y se convierten a NaN:

- `000`: ausencia de tumor o masa
- `989`: tumor de 989 mm o más (valor censurado)
- `990`: foco microscópico sin tamaño especificado
- `991-995`: tamaños aproximados (menos de 1 a 5 cm)
- `996-998`: códigos administrativos sin valor de tamaño
- `999`: tamaño desconocido o no registrado

Fuente: SEER Training Modules — Coding CS Tumor Size. National Cancer Institute. training.seer.cancer.gov/collaborative/system/tnm/t/size/

In [15]:
# Numérica continua en mm

codigos_especiales = ['000', '989', '990', '991', '992', '993', '994', '995', '996', '997', '998', '999']

df['tumor_size_cs'] = df['tumor_size_cs'].replace(codigos_especiales, np.nan)
df['tumor_size_cs'] = pd.to_numeric(df['tumor_size_cs'], errors='coerce')

print(f"Dtype: {df['tumor_size_cs'].dtype}")
print(f"NaN totales: {df['tumor_size_cs'].isnull().sum()} ({df['tumor_size_cs'].isnull().mean()*100:.1f}%)")

Dtype: float64
NaN totales: 2636 (59.4%)


#### surgery_code

Los códigos numéricos de `RX Summ--Surg Prim Site (1998-2022)` se agrupan en 4 categorías clínicas siguiendo la metodología de estudios poblacionales recientes con datos SEER para tumores óseos (Carron et al., 2026):

- `No surgery` → código 00
- `Partial resection` → códigos 15, 19, 20, 25, 26, 27
- `Radical resection` → códigos 40, 41, 42, 50, 51, 52, 53, 54, 55, 60, 61
- `Surgery NOS / Unknown` → códigos 90, 98, 99 y NaN

Fuente: National Cancer Institute, SEER Program.
*Surgery to Primary Site — RX Summ–Surg Prim Site (1998-2022)*.
Available at: seer.cancer.gov/seerstat/variables/seer/surgery/

Referencia metodológica: Carron CJ, et al. *North American Spine Society Journal*,
2026. DOI: 10.1016/j.nassj.2026.100865

In [16]:
# Agrupación en 4 categorías clínicas

surgery_map = {
    '00': 'No surgery',
    '10': 'No surgery',
    '15': 'Partial resection',
    '19': 'Partial resection',
    '20': 'Partial resection',
    '21': 'Partial resection',
    '22': 'Partial resection',
    '25': 'Partial resection',
    '26': 'Partial resection',
    '27': 'Partial resection',
    '37': 'Partial resection',
    '40': 'Radical resection',
    '41': 'Radical resection',
    '42': 'Radical resection',
    '50': 'Radical resection',
    '51': 'Radical resection',
    '52': 'Radical resection',
    '53': 'Radical resection',
    '54': 'Radical resection',
    '55': 'Radical resection',
    '60': 'Radical resection',
    '61': 'Radical resection',
    '30': 'Radical resection',
    '90': 'Surgery NOS / Unknown',
    '98': 'Surgery NOS / Unknown',
    '99': 'Surgery NOS / Unknown',
}

df['surgery_code'] = df['surgery_code'].map(surgery_map)
df['surgery_code'] = pd.Categorical(df['surgery_code'])

print(df['surgery_code'].value_counts(dropna=False))

surgery_code
Radical resection        2783
No surgery                868
Partial resection         555
NaN                       122
Surgery NOS / Unknown     108
Name: count, dtype: int64


#### radiation

In [17]:
# Agrupación en 4 categorías clínicas

radiation_map = {
    'None/Unknown': 'No radiation',
    'Beam radiation': 'Radiation',
    'Radiation, NOS  method or source not specified': 'Radiation',
    'Radioactive implants (includes brachytherapy) (1988+)': 'Radiation',
    'Combination of beam with implants or isotopes': 'Radiation',
    'Radioisotopes (1988+)': 'Radiation',
    'Refused (1988+)': 'Refused',
    'Recommended, unknown if administered': 'Recommended / Unknown',
}

df['radiation'] = df['radiation'].map(radiation_map)
df['radiation'] = pd.Categorical(df['radiation'])

print(df['radiation'].value_counts(dropna=False))

radiation
No radiation             3462
Radiation                 931
Recommended / Unknown      28
Refused                    15
Name: count, dtype: int64


#### chemotherapy

In [18]:
# Categórica nominal con 2 valores
df['chemotherapy'] = pd.Categorical(df['chemotherapy'])

print(df['chemotherapy'].value_counts(dropna=False))

chemotherapy
Yes           4201
No/Unknown     235
Name: count, dtype: int64


#### survival_months_flag

In [19]:
# Simplificación a códigos cortos
flag_map = {
    'Complete dates are available and there are more than 0 days of survival': 'complete',
    'Complete dates are available and there are 0 days of survival': 'complete_zero',
    'Incomplete dates are available and there cannot be zero days of follow-up': 'incomplete',
    'Incomplete dates are available and there could be zero days of follow-up': 'incomplete_zero',
}

df['survival_months_flag'] = df['survival_months_flag'].map(flag_map)
df['survival_months_flag'] = pd.Categorical(df['survival_months_flag'])

print(df['survival_months_flag'].value_counts(dropna=False))

survival_months_flag
complete           4247
incomplete          184
complete_zero         4
incomplete_zero       1
Name: count, dtype: int64


> **Nota:** Los 185 casos con fechas incompletas (`incomplete` e `incomplete_zero`) representan el 4.2% del dataset. Sus valores de `survival_months` son estimaciones con posible margen de error. Se mantienen en el dataset pero se tendrán en cuenta en el análisis de supervivencia del EDA.

#### cause_specific_death

In [20]:
# Simplificación a códigos cortos

cause_map = {
    'Alive or dead of other cause': 'Alive / Other cause',
    'Dead (attributable to this cancer dx)': 'Dead (cancer)',
    'Dead (missing/unknown COD)': 'Dead (unknown cause)',
}

df['cause_specific_death'] = df['cause_specific_death'].map(cause_map)
df['cause_specific_death'] = pd.Categorical(df['cause_specific_death'])

print(df['cause_specific_death'].value_counts(dropna=False))

cause_specific_death
Alive / Other cause     2985
Dead (cancer)           1422
Dead (unknown cause)      29
Name: count, dtype: int64


> **Nota sobre data leakage:** Las columnas `cause_specific_death`, `vital_status`, `survival_months` y `survival_months_flag` son variables de seguimiento posterior al diagnóstico. Se conservan en el dataset para el EDA y para construir la variable objetivo, pero quedarán excluidas del conjunto de features (X) en el notebook de modelado.

#### vital_status

In [21]:
# Categórica nominal, con 2 valores
df['vital_status'] = pd.Categorical(df['vital_status'])

print(df['vital_status'].value_counts(dropna=False))

vital_status
Alive    2850
Dead     1586
Name: count, dtype: int64


### 5.4) Conjunto de datos transformado

In [24]:
df.head(5)

,age_group,sex,year_diagnosis,histology_code,primary_site,stage,tumor_size_cs,surgery_code,radiation,chemotherapy,survival_months,survival_months_flag,cause_specific_death,vital_status,tumor_type
0,05-09 years,Female,2004,9260,Upper limb bones,Regional by direct extension only,64.0,Partial resection,Radiation,Yes,239,complete,Alive / Other cause,Alive,Ewing sarcoma
1,15-19 years,Male,2001,9260,Trunk / ribs / sternum,NaN,NaN,Radical resection,No radiation,Yes,275,complete,Alive / Other cause,Alive,Ewing sarcoma
2,10-14 years,Female,2001,9260,Pelvis / hip,NaN,NaN,No surgery,Radiation,Yes,119,complete,Dead (cancer),Dead,Ewing sarcoma
3,10-14 years,Female,2001,9180,Lower limb bones,NaN,NaN,Radical resection,No radiation,Yes,274,complete,Alive / Other cause,Alive,Osteosarcoma NOS
4,15-19 years,Female,2000,9181,Pelvis / hip,NaN,NaN,No surgery,Radiation,Yes,23,complete,Dead (cancer),Dead,Chondroblastic osteosarcoma


In [25]:
print("---- VALORES ÚNICOS POR COLUMNA TRAS CAMBIOS ----")
for col in df.columns:
    n_unique = df[col].nunique()
    sample = df[col].dropna().unique()
    print(f"\n{col} (dtype: {df[col].dtype}) — {n_unique} valores únicos")
    print(f"  Valores:\n {sample}")

---- VALORES ÚNICOS POR COLUMNA TRAS CAMBIOS ----

age_group (dtype: category) — 5 valores únicos
  Valores:
 ['05-09 years', '15-19 years', '10-14 years', '00 years', '01-04 years']
Categories (5, object): ['00 years' < '01-04 years' < '05-09 years' < '10-14 years' < '15-19 years']

sex (dtype: category) — 2 valores únicos
  Valores:
 ['Female', 'Male']
Categories (2, object): ['Female', 'Male']

year_diagnosis (dtype: int64) — 24 valores únicos
  Valores:
 [2004 2001 2000 2002 2003 2005 2006 2007 2012 2008 2009 2010 2011 2022
 2013 2014 2015 2020 2016 2017 2018 2019 2021 2023]

histology_code (dtype: int64) — 9 valores únicos
  Valores:
 [9260 9180 9181 9186 9182 9183 9187 9185 9184]

primary_site (dtype: category) — 8 valores únicos
  Valores:
 ['Upper limb bones', 'Trunk / ribs / sternum', 'Pelvis / hip', 'Lower limb bones', 'Vertebral column', 'Skull / head / jaw', 'Other / NOS', 'CNS / meninges']
Categories (8, object): ['CNS / meninges', 'Lower limb bones', 'Other / NOS', 'Pelvi